# Fase 10 — **k5-Aug** (desain ADITIF) dengan resep `optimizer="auto"`

Menutup cacat desain yang sudah terdokumentasi sejak lama tapi **tidak pernah dijalankan**.

| | P1 (`k3_replace`) | **k5-Aug (`k5_augment`)** |
|---|---|---|
| Baseline SS2D | K=4 (cross-scan) | K=4 (cross-scan) |
| Varian | K=3 (fwd+bwd+deformable) | **K=5 (4 arah + deformable)** |
| Sifat | **SUBSTITUTIF** — 2 jalur kolom dibuang | **ADITIF** — tidak ada yang dibuang |
| `ssm_ratio` | 1.4 (handicap SSM 30%) | **1.7** (handicap 15%) |
| `ConvOffset.proj` | ada | **dibuang** (lebih setia DefMamba) |
| Parameter | 5.930.869 (−0,94%) | **6.215.245 (+3,81%)** — lolos ±5% |

Spesifikasi mengikuti `PATCH-k5aug.md` **persis**. Seluruh rantai patch sudah diuji offline:
parameter keluar **tepat 6.215.245**, 10 modul DeformSS2D, **K=5** — dan P1/B2 lama tetap
reproducible (semua argumen baru default ke perilaku lama).

### ⚠️ Resep: `optimizer="auto"`, BUKAN `lr0=0.002`
Fase 9 membuktikan resep terkunci merugikan **setiap** model 7–11 poin (B2: 0.5698 → **0.6796**).
Membangun varian baru lalu mengujinya di resep itu = mengulang kesalahan yang baru ditemukan.
Pembanding yang benar adalah **B2-auto = 0.6796**, bukan B2 = 0.5662.

**Ujian tambahan yang berharga:** P1 **divergen** di AdamW. Kalau k5-Aug bisa dilatih di sini,
itu temuan tersendiri — bahkan sebelum bicara akurasi.

**+ Pemantau offset per-epoch** (sel 7b) — hanya membaca; keluaran: `offset_monitor.csv`.

## 1. Mount Drive + Kredensial Kaggle

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import os, sys, glob, shutil, subprocess
DRIVE_ROOT   = "/content/drive/MyDrive/demamba-yolowd"
DRIVE_OUTPUT = f"{DRIVE_ROOT}/output"
os.makedirs(DRIVE_OUTPUT, exist_ok=True); os.makedirs(f"{DRIVE_ROOT}/analysis", exist_ok=True)
os.makedirs("/root/.kaggle", exist_ok=True)
shutil.copy(f"{DRIVE_ROOT}/kaggle.json", "/root/.kaggle/kaggle.json"); os.chmod("/root/.kaggle/kaggle.json", 0o600)
subprocess.run([sys.executable,"-m","pip","install","-q","kaggle"], check=True)
print("Drive + Kaggle siap")

Mounted at /content/drive
Drive + Kaggle siap


## 2. Dataset + GPU

In [ ]:
import torch
DATA_DIR = "/content/data"
if not glob.glob(f"{DATA_DIR}/**/*.txt", recursive=True):
    os.makedirs(DATA_DIR, exist_ok=True)
    subprocess.run(["kaggle","datasets","download","-d","benyxu/dsf-yolo-steel-tube-dataset",
                    "-p",DATA_DIR,"--unzip"], check=True)
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "TIDAK ADA")
print(f"cpu_count={os.cpu_count()} -> workers={min(os.cpu_count()//max(torch.cuda.device_count(),1),8)}")
assert torch.cuda.is_available(), "GPU tidak aktif -- Runtime > Change runtime type"

GPU: NVIDIA A100-SXM4-40GB
cpu_count=12 -> workers=8


## 3. 🔧 Konfigurasi

In [ ]:
SEED, TARGET_EPOCHS = 1, 150
OPTIMIZER   = "auto"          # resep wajar (Fase 9). lr0 & momentum DIABAIKAN oleh "auto"
USE_COS_LR  = False
MODEL_LABEL = "K5-auto"
RUN_NAME    = f"k5aug_seed{SEED}_{TARGET_EPOCHS}ep_auto"
EXPECTED_PARAMS = 6_215_245   # dari PATCH-k5aug.md, diverifikasi offline
DEFORM_SRC  = f"{DRIVE_ROOT}/code/deform_ss2d.py"   # <- versi SUDAH DIPERBAIKI grid-nya

print(f"varian   : k5-Aug (k5_augment, ssm_ratio=1.7, use_proj=False)")
print(f"RUN_NAME : {RUN_NAME}   label CSV: {MODEL_LABEL}")
print(f"resep    : optimizer=auto -> AdamW lr=0.000833 (iterations 6450 <= 10000)")
print(f"pembanding: B2-auto 0.6796 | YOLOv8n-auto 0.6639 | YOLOv8s-auto 0.6980")
print(f"           (P1 DIVERGEN di AdamW -- kalau k5-Aug bertahan, itu temuan)")

varian   : k5-Aug (k5_augment, ssm_ratio=1.7, use_proj=False)
RUN_NAME : k5aug_seed1_150ep_auto   label CSV: K5-auto
resep    : optimizer=auto -> AdamW lr=0.000833 (iterations 6450 <= 10000)
pembanding: B2-auto 0.6796 | YOLOv8n-auto 0.6639 | YOLOv8s-auto 0.6980
           (P1 DIVERGEN di AdamW -- kalau k5-Aug bertahan, itu temuan)


## 4. Clone + Patch + Build Kernel

Urutan patch: **1.x** (SSM_CLASS) → **2.1** (impor) → **3.x** (registrasi P1) → **4.x/5.1**
(`torch.load`) → **K5-1..4** (`use_proj` + wrapper) → **K5-5** (registrasi K5) → **K5-6** (YAML).

`deform_ss2d.py` diambil dari Drive (**versi yang grid-nya sudah diperbaiki**), bukan dari repo.

In [ ]:
import re, py_compile
REPO = "/content/Mamba-YOLO"
os.chdir("/content")
if os.path.exists(REPO): shutil.rmtree(REPO)
subprocess.run(["git","clone","--depth","1","https://github.com/HZAI-ZJNU/Mamba-YOLO.git",REPO], check=True)

assert os.path.exists(DEFORM_SRC), f"letakkan deform_ss2d.py (versi grid diperbaiki) di {DEFORM_SRC}"
shutil.copy(DEFORM_SRC, f"{REPO}/ultralytics/nn/modules/deform_ss2d.py")
import hashlib
print("sha256 deform_ss2d.py:", hashlib.sha256(open(DEFORM_SRC,"rb").read()).hexdigest()[:16],
      "(versi perbaikan grid = 67544587...)")

def patch(path, subs):
    p = f"{REPO}/{path}"; src = open(p).read()
    for old, new, tag in subs:
        if new in src: print(f"  ~ {tag}: skip"); continue
        assert src.count(old) == 1, f"GAGAL {tag}: {src.count(old)}x (harus 1x)"
        src = src.replace(old, new); print(f"  OK {tag}")
    open(p, "w").write(src); py_compile.compile(p, doraise=True)

M, D, T = "ultralytics/nn/modules/mamba_yolo.py", "ultralytics/nn/modules/deform_ss2d.py", "ultralytics/nn/tasks.py"
print("### 1.x registrasi SSM_CLASS")
patch(M, [("class XSSBlock(nn.Module):\n    def __init__(","class XSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(","1.1"),
 ("nn.Sequential(*(SS2D(d_model=self.hidden_dim,","nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,","1.2"),
 ("class VSSBlock(nn.Module):\n    def __init__(","class VSSBlock(nn.Module):\n    SSM_CLASS = SS2D\n    def __init__(","1.3"),
 ("            self.op = SS2D(","            self.op = self.SSM_CLASS(","1.4")])
print("### 2.1 + K5-1..3 (use_proj)")
patch(D, [("from .mamba_yolo import SS2D","from .mamba_yolo import SS2D, VSSBlock, XSSBlock","2.1"),
 ('    def __init__(self, in_ch: int, kernel_size: int):\n        super().__init__()\n        pad = kernel_size // 2 if kernel_size > 1 else 0\n        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1)',
  '    def __init__(self, in_ch: int, kernel_size: int, use_proj: bool = True):\n        super().__init__()\n        pad = kernel_size // 2 if kernel_size > 1 else 0\n        self.use_proj = use_proj\n        self.proj = nn.Conv2d(in_ch, in_ch, kernel_size=1) if use_proj else nn.Identity()', "K5-1"),
 ('    def __init__(self, embed_dim: int, kernel_size: int = 5):\n        super().__init__()\n        self.conv_offset = ConvOffset(embed_dim, kernel_size)',
  '    def __init__(self, embed_dim: int, kernel_size: int = 5, use_proj: bool = True):\n        super().__init__()\n        self.conv_offset = ConvOffset(embed_dim, kernel_size, use_proj=use_proj)', "K5-2"),
 ('                 deform_mode: str = "k3_replace",\n                 deform_stage: int = 0,\n                 deform_kernel_size: int = None,\n                 **kwargs):',
  '                 deform_mode: str = "k3_replace",\n                 deform_stage: int = 0,\n                 deform_kernel_size: int = None,\n                 deform_use_proj: bool = True,\n                 **kwargs):', "K5-3a"),
 ('DeformableScan(d_inner, kernel_size=ks) for _ in range(n_deform)',
  'DeformableScan(d_inner, kernel_size=ks, use_proj=deform_use_proj) for _ in range(n_deform)', "K5-3b")])

print("### K5-4 wrapper k5-Aug")
K5 = '''

class DeformVSSBlockK5(VSSBlock):
    """VSSBlock varian k5-Aug (ADITIF, K=4 -> K=5). ssm_ratio 1.7, tanpa ConvOffset.proj."""
    SSM_CLASS = partial(DeformSS2D, deform_mode="k5_augment",
                        deform_kernel_size=5, deform_use_proj=False)
    def __init__(self, *args, ssm_ratio=2.0, **kwargs):
        super().__init__(*args, ssm_ratio=1.7, **kwargs)


class DeformXSSBlockK5(XSSBlock):
    """XSSBlock varian k5-Aug - neck. Lihat DeformVSSBlockK5."""
    SSM_CLASS = partial(DeformSS2D, deform_mode="k5_augment",
                        deform_kernel_size=5, deform_use_proj=False)
    def __init__(self, *args, ssm_ratio=2.0, **kwargs):
        super().__init__(*args, ssm_ratio=1.7, **kwargs)
'''
_p = f"{REPO}/{D}"
if "DeformVSSBlockK5" not in open(_p).read():
    open(_p, "a").write(K5); print("  OK K5-4")
else: print("  ~ K5-4: skip")
py_compile.compile(_p, doraise=True)

print("### 3.x + K5-5 registrasi (P1 DAN K5 -- P1 wajib ikut agar regresi aman)")
patch(T, [("    XSSBlock\n)\n",
  "    XSSBlock\n)\nfrom ultralytics.nn.modules.deform_ss2d import (DeformVSSBlock, DeformXSSBlock,\n                                                DeformVSSBlockK5, DeformXSSBlockK5)\n","3.1+K5-5"),
 ("SimpleStem, VisionClueMerge, VSSBlock, XSSBlock\n        }:",
  "SimpleStem, VisionClueMerge, VSSBlock, XSSBlock, DeformVSSBlock, DeformXSSBlock, DeformVSSBlockK5, DeformXSSBlockK5\n        }:","3.2+K5-5b"),
 ("RepC3, XSSBlock}:","RepC3, XSSBlock, DeformXSSBlock, DeformXSSBlockK5}:","3.3+K5-5c"),
 ('            ckpt = torch.load(file, map_location="cpu")\n\n    except ModuleNotFoundError as e:',
  '            ckpt = torch.load(file, map_location="cpu", weights_only=False)\n\n    except ModuleNotFoundError as e:',"4.1"),
 ('        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu")',
  '        check_requirements(e.name)  # install missing module\n        ckpt = torch.load(file, map_location="cpu", weights_only=False)',"4.2")])
patch("ultralytics/utils/torch_utils.py", [
 ('    x = torch.load(f, map_location=torch.device("cpu"))\n    if "model" not in x:',
  '    x = torch.load(f, map_location=torch.device("cpu"), weights_only=False)\n    if "model" not in x:',"5.1")])

print("### K5-6 YAML")
_y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
_y = re.sub(r"\bVSSBlock\b","DeformVSSBlockK5",_y); _y = re.sub(r"\bXSSBlock\b","DeformXSSBlockK5",_y)
K5_YAML = f"{REPO}/ultralytics/cfg/models/mamba-yolo/mamba-yolo-T-k5aug.yaml"
open(K5_YAML,"w").write(_y); print("  OK", os.path.basename(K5_YAML))

# ---- kernel selective_scan (cache di Drive) ----
subprocess.run([sys.executable,"-m","pip","install","-q","ninja"], check=True)
os.environ["MAX_JOBS"] = str(max(1,int(subprocess.run(["nproc"],capture_output=True,text=True).stdout.strip() or "2")))
cap = torch.cuda.get_device_capability(0)
SS_TAG = f"sm{cap[0]}{cap[1]}_torch{torch.__version__.split('+')[0]}_py{sys.version_info.major}{sys.version_info.minor}"
SS_DIR = f"{DRIVE_ROOT}/wheels/selective_scan_{SS_TAG}"; os.makedirs(SS_DIR, exist_ok=True)
whl = glob.glob(f"{SS_DIR}/*.whl")
if whl:
    print(f"Wheel cache ({SS_TAG})"); subprocess.run([sys.executable,"-m","pip","install"]+whl, check=True)
else:
    print(f"Compile kernel sekali ({SS_TAG})")
    subprocess.run([sys.executable,"-m","pip","wheel",".","--no-deps","-v","-w",SS_DIR], cwd=f"{REPO}/selective_scan", check=True)
    subprocess.run([sys.executable,"-m","pip","install"]+glob.glob(f"{SS_DIR}/*.whl"), check=True)
subprocess.run([sys.executable,"-m","pip","install","-q","-e","."], cwd=REPO, check=True)
for k in [x for x in list(sys.modules) if x.startswith("ultralytics")]: del sys.modules[k]
import selective_scan_cuda_core; print("selective_scan_cuda_core OK")
subprocess.run([sys.executable,"-m","pip","install","-q","seaborn","thop","timm","einops"], check=True)
subprocess.run([sys.executable,"-m","pip","uninstall","-y","-q","wandb","ray","tensorboard","tensorboardX",
                "tensorboard-data-server","clearml","comet_ml","dvclive","mlflow"], check=False)
print("Lingkungan siap")

sha256 deform_ss2d.py: 675445871a2e4d55 (versi perbaikan grid = 67544587...)
### 1.x registrasi SSM_CLASS
  OK 1.1
  OK 1.2
  OK 1.3
  OK 1.4
### 2.1 + K5-1..3 (use_proj)
  ~ 2.1: skip
  OK K5-1
  OK K5-2
  OK K5-3a
  OK K5-3b
### K5-4 wrapper k5-Aug
  OK K5-4
### 3.x + K5-5 registrasi (P1 DAN K5 -- P1 wajib ikut agar regresi aman)
  OK 3.1+K5-5
  OK 3.2+K5-5b
  OK 3.3+K5-5c
  OK 4.1
  OK 4.2
  OK 5.1
### K5-6 YAML
  OK mamba-yolo-T-k5aug.yaml
Wheel cache (sm80_torch2.11.0_py313)
selective_scan_cuda_core OK
Lingkungan siap


## 5. data.yaml — identik dgn 40 run DSF sebelumnya

In [ ]:
import yaml
from pathlib import Path
cands = [p.parent.parent for p in Path(DATA_DIR).rglob("images/train2021") if p.is_dir()]
assert cands, f"struktur images/train2021 tidak ditemukan di {DATA_DIR}"
DSF_BASE = str(cands[0].resolve())
NAMES = ["air-hole","bite-edge","broken-arc","crack","dense-gas-hole","non-fusion","slag-inclusion","unfused-hole"]
DATA_YAML = f"{REPO}/ultralytics/cfg/datasets/dsf-yolo.yaml"
yaml.safe_dump({"path":DSF_BASE,"train":"images/train2021","val":"images/val2021",
                "nc":len(NAMES),"names":NAMES}, open(DATA_YAML,"w"), sort_keys=False)
for s in ["train2021","val2021"]:
    d=Path(DSF_BASE)/"images"/s
    print(f"  {s}: {len([f for f in d.iterdir() if f.suffix.lower() in {'.jpg','.jpeg','.png','.bmp'}])} citra")
assert (Path(DSF_BASE)/"labels"/"train2021").is_dir(), "folder labels/ tidak ada"
print(f"\nDSF_BASE: {DSF_BASE}")

  train2021: 2726 citra
  val2021: 682 citra

DSF_BASE: /content/data/steel-tube-dataset-all/yolo


## 6. Pre-flight WAJIB — **hentikan kalau angkanya meleset**

Tiga hal dicek: k5-Aug = 6.215.245 & K=5 & 10 modul, **dan** regresi P1/B2 lama tetap utuh.

In [ ]:
os.chdir(REPO)
from copy import deepcopy
from ultralytics import YOLO
from ultralytics.nn.tasks import parse_model, yaml_model_load
from ultralytics.nn.modules.deform_ss2d import DeformSS2D

d = yaml_model_load(K5_YAML); d["nc"] = len(NAMES)
m,_ = parse_model(deepcopy(d), ch=3, verbose=False)
n_par = sum(p.numel() for p in m.parameters())
ds = [x for x in m.modules() if isinstance(x, DeformSS2D)]
print(f"k5-Aug : {n_par:,} parameter (harap {EXPECTED_PARAMS:,})   Δ vs B2 = {(n_par-5_986_912)/5_986_912*100:+.2f}%")
print(f"         {len(ds)} modul DeformSS2D (harap 10), K = {sorted({x.K for x in ds})} (harap [5])")
assert n_par == EXPECTED_PARAMS, "PARAMETER MELESET -- JANGAN mulai training"
assert len(ds) == 10 and {x.K for x in ds} == {5}, "mode/jumlah modul salah"
del m, ds

# regresi: P1 & B2 lama harus tetap utuh
_y = open(f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml").read()
_p1 = re.sub(r"\bXSSBlock\b","DeformXSSBlock", re.sub(r"\bVSSBlock\b","DeformVSSBlock",_y))
open("/content/p1_regresi.yaml","w").write(_p1)
for lbl,f,exp in [("P1 lama","/content/p1_regresi.yaml",5_930_869),
                  ("B2 lama",f"{REPO}/ultralytics/cfg/models/mamba-yolo/Mamba-YOLO-T.yaml",5_986_912)]:
    dd = yaml_model_load(f); dd["nc"] = len(NAMES)
    mm,_ = parse_model(deepcopy(dd), ch=3, verbose=False)
    pp = sum(p.numel() for p in mm.parameters()); del mm
    print(f"  regresi {lbl}: {pp:,} (harap {exp:,}) -> {'AMAN' if pp==exp else 'RUSAK'}")
    assert pp == exp, f"patch merusak {lbl} -- hentikan"
print("\nPre-flight LOLOS")

WARNING ⚠️ no model scale passed. Assuming scale='T'.


/usr/local/lib/python3.13/dist-packages/timm/models/layers/__init__.py:49: FutureWarning: Importing from timm.models.layers is deprecated, please import via timm.layers
  warnings.warn(f"Importing from {__name__} is deprecated, please import via timm.layers", FutureWarning)
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:103: FutureWarning: `torch.cuda.amp.custom_fwd(args...)` is deprecated. Please use `torch.amp.custom_fwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_fwd
/content/Mamba-YOLO/ultralytics/nn/modules/common_utils_mbyolo.py:130: FutureWarning: `torch.cuda.amp.custom_bwd(args...)` is deprecated. Please use `torch.amp.custom_bwd(args..., device_type='cuda')` instead.
  @torch.cuda.amp.custom_bwd


k5-Aug : 6,215,245 parameter (harap 6,215,245)   Δ vs B2 = +3.81%
         10 modul DeformSS2D (harap 10), K = [5] (harap [5])
WARNING ⚠️ no model scale passed. Assuming scale='T'.


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


  regresi P1 lama: 5,930,869 (harap 5,930,869) -> AMAN
WARNING ⚠️ no model scale passed. Assuming scale='T'.
  regresi B2 lama: 5,986,912 (harap 5,986,912) -> AMAN

Pre-flight LOLOS


## 7. Status checkpoint

In [ ]:
last_ckpt = Path(DRIVE_OUTPUT)/RUN_NAME/"weights"/"last.pt"
if not last_ckpt.exists():
    STATUS="BARU"; print("STATUS: BARU")
else:
    raw = torch.load(str(last_ckpt), map_location="cpu", weights_only=False)
    ep, ho = raw.get("epoch",-1), raw.get("optimizer") is not None
    STATUS = "BELUM_SELESAI" if (ho and 0 <= ep < TARGET_EPOCHS-1) else "SELESAI"
    print(f"STATUS: {STATUS} (epoch={ep}, optimizer={ho})")

STATUS: BARU


## 7b. Pemantau offset (HANYA MEMBACA)

Mencatat per epoch, pada batch TRAINING nyata: bias & norma bobot offset network, `tanh(Δp)` rata-rata,
faktor gradien `(1−tanh²)`, % token di luar [−1,1], dan % token yang urutannya berpindah — ke
`offset_monitor.csv` di folder run. Latar belakang: pada checkpoint P1-fix path offset macet jenuh di ±1
(urutan token ≈ identitas di peta terbesar). Pemantau ini menunjukkan **kapan** hal itu terjadi di k5-Aug.

Diuji pada training Ultralytics sungguhan (CPU, 2 epoch): loss dengan vs tanpa pemantau **identik persis**;
hook mengembalikan `None` (tidak mengubah keluaran); tidak ada `.item()` per langkah.

In [ ]:
import csv
import torch
from pathlib import Path

class OffsetMonitor:
    """Memantau offset network per epoch pada batch TRAINING nyata.
    HANYA MEMBACA: hook mengembalikan None (keluaran tidak diubah), tanpa .item() per langkah."""
    FIELDS = ["epoch", "block", "N", "bias_dy", "bias_dx", "bias_dp", "w_dy", "w_dx", "w_dp",
              "off_px", "spatial_sat", "tanh_p_mean", "tanh_p_absmean", "tanh_p_std_tok",
              "tanh_grad", "clamp_frac", "moved_frac", "disp_mean", "disp_max", "n_fwd"]

    def __init__(self):
        self.handles, self.mods, self.acc, self.n, self.last = [], {}, {}, {}, {}

    def attach(self, model):
        from ultralytics.nn.modules.deform_ss2d import DeformSS2D
        for h in self.handles: h.remove()
        self.handles, self.mods, self.acc, self.n, self.last = [], {}, {}, {}, {}
        for name, m in model.named_modules():
            if isinstance(m, DeformSS2D):
                for j, ds in enumerate(m.deform_scans):
                    key = f"{name}.{j}"; out = ds.conv_offset.out
                    self.mods[key] = out
                    self.handles.append(out.register_forward_hook(self._make_hook(key)))
        print(f"[OffsetMonitor] terpasang pada {len(self.mods)} offset network")

    def _make_hook(self, key):
        def hook(mod, inp, out):
            if not mod.training:
                return None                                   # validasi: abaikan
            with torch.no_grad():
                z = out.detach().float()                      # (B,3,H,W)
                N = z.shape[2] * z.shape[3]
                ty, tx = torch.tanh(z[:, 0]), torch.tanh(z[:, 1])
                tp = torch.tanh(z[:, 2]).flatten(1)           # (B,N)
                ref = torch.linspace(-1, 1, N, device=z.device)
                v = torch.stack([
                    0.25 * (ty.abs().mean() + tx.abs().mean()),               # |offset| spasial, piksel
                    ((ty.abs() > 0.99) | (tx.abs() > 0.99)).float().mean(),   # jenuh di batas spasial
                    tp.mean(), tp.abs().mean(),                                # tanh(dp): rata2 & |rata2|
                    tp.std(dim=1).mean(),                                      # variasi antar-token
                    (1 - tp ** 2).mean(),                                      # faktor gradien tanh
                    ((ref[None] + tp).abs() > 1).float().mean(),               # token di luar [-1,1]
                ])
                self.acc[key] = self.acc.get(key, 0) + v                       # tetap di GPU
                self.n[key] = self.n.get(key, 0) + 1
                self.last[key] = out.detach()[:1]                              # satu citra utk statistik urutan
            return None                                                        # JANGAN mengubah keluaran
        return hook

    def flush(self, epoch, run_dir):
        rows = []
        for key, out in self.mods.items():
            n = self.n.get(key, 0)
            if n == 0: continue
            a = (self.acc[key] / n).tolist()
            w = out.weight.detach().float().flatten(1); b = out.bias.detach().float()
            z = self.last[key].float(); N = z.shape[2] * z.shape[3]
            tp = torch.tanh(z[:, 2]).flatten(1); ref = torch.linspace(-1, 1, N, device=z.device)
            idx = torch.argsort(ref[None] + tp, dim=-1, stable=True)
            d = (idx - torch.arange(N, device=z.device)[None]).abs().float()
            rows.append(dict(epoch=epoch, block=key, N=N,
                bias_dy=b[0].item(), bias_dx=b[1].item(), bias_dp=b[2].item(),
                w_dy=w[0].norm().item(), w_dx=w[1].norm().item(), w_dp=w[2].norm().item(),
                off_px=a[0], spatial_sat=a[1], tanh_p_mean=a[2], tanh_p_absmean=a[3],
                tanh_p_std_tok=a[4], tanh_grad=a[5], clamp_frac=a[6],
                moved_frac=(idx != torch.arange(N, device=z.device)[None]).float().mean().item(),
                disp_mean=d.mean().item(), disp_max=d.max().item(), n_fwd=n))
        if rows:
            path = Path(run_dir) / "offset_monitor.csv"
            new = not path.exists()
            with open(path, "a", newline="") as f:
                wr = csv.DictWriter(f, fieldnames=self.FIELDS)
                if new: wr.writeheader()
                wr.writerows(rows)
            if epoch == 1 or epoch % 10 == 0:
                sat = sum(abs(r["tanh_p_mean"]) > 0.99 for r in rows)
                print(f"[OffsetMonitor] epoch {epoch}: blok |tanh(dp)| rata2>0.99: {sat}/{len(rows)} | "
                      f"token di luar [-1,1]: {sum(r['clamp_frac'] for r in rows)/len(rows):.1%} | "
                      f"token pindah: {sum(r['moved_frac'] for r in rows)/len(rows):.1%}")
        self.acc, self.n = {}, {}


## 8. Training

In [ ]:
torch.backends.cuda.matmul.allow_tf32 = False
torch.backends.cudnn.allow_tf32 = False

monitor = OffsetMonitor()
def _attach(tr): monitor.attach(tr.model)                       # dipasang di awal training (juga saat resume)
def _flush(tr):  monitor.flush(tr.epoch + 1, tr.save_dir)       # tulis statistik offset tiap epoch

if STATUS == "BARU":
    model = YOLO(K5_YAML)
    model.add_callback("on_train_start", _attach); model.add_callback("on_train_epoch_end", _flush)
    model.train(data=DATA_YAML, epochs=TARGET_EPOCHS,
        batch=8, imgsz=640, device=0, amp=True, seed=SEED,
        patience=0, cache=True, deterministic=True, workers=8,
        optimizer=OPTIMIZER, cos_lr=USE_COS_LR,
        project=DRIVE_OUTPUT, name=RUN_NAME, plots=False)
elif STATUS == "BELUM_SELESAI":
    model = YOLO(str(last_ckpt))
    model.add_callback("on_train_start", _attach); model.add_callback("on_train_epoch_end", _flush)
    model.train(resume=True)
else:
    model = YOLO(str(last_ckpt)); print("Sudah selesai -- tidak training ulang.")

SAVE_DIR = model.trainer.save_dir if getattr(model,"trainer",None) else Path(DRIVE_OUTPUT)/RUN_NAME
print(f"\nSAVE_DIR: {SAVE_DIR}")
print("CATAT baris 'optimizer:' di log (AdamW + lr yang dipilih framework).")
print("CATAT juga: apakah cls_loss tetap sehat? P1 DIVERGEN di resep ini.")
fin = torch.load(str(Path(SAVE_DIR)/"weights"/"last.pt"), map_location="cpu", weights_only=False)
print("final_eval selesai" if fin.get("optimizer") is None else f"BELUM tuntas (epoch={fin.get('epoch')}) -- jalankan ulang utk resume")

WARNING ⚠️ no model scale passed. Assuming scale='T'.
New https://pypi.org/project/ultralytics/8.4.166 available 😃 Update with 'pip install -U ultralytics'
Ultralytics YOLOv8.2.29 🚀 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (NVIDIA A100-SXM4-40GB, 40441MiB)
engine/trainer: task=detect, mode=train, model=/content/Mamba-YOLO/ultralytics/cfg/models/mamba-yolo/mamba-yolo-T-k5aug.yaml, data=/content/Mamba-YOLO/ultralytics/cfg/datasets/dsf-yolo.yaml, epochs=150, time=None, patience=0, batch=8, imgsz=640, save=True, save_period=-1, cache=True, device=0, workers=8, project=/content/drive/MyDrive/demamba-yolowd/output, name=k5aug_seed1_150ep_auto, exist_ok=False, pretrained=True, optimizer=auto, verbose=True, seed=1, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=10, resume=False, amp=True, fraction=1.0, profile=False, freeze=None, multi_scale=False, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou

100%|██████████| 755k/755k [00:00<00:00, 172MB/s]

Overriding model.yaml nc=80 with nc=8
WARNING ⚠️ no model scale passed. Assuming scale='T'.

                   from  n    params  module                                       arguments                     
  0                  -1  1      5136  ultralytics.nn.modules.mamba_yolo.SimpleStem [3, 32, 3]                    
  1                  -1  1     36516  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockK5[32, 32]                      
  2                  -1  1      8384  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[32, 64]                      
  3                  -1  1    111046  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockK5[64, 64]                      
  4                  -1  1     33152  ultralytics.nn.modules.mamba_yolo.VisionClueMerge[64, 128]                     
  5                  -1  3   1124949  ultralytics.nn.modules.deform_ss2d.DeformVSSBlockK5[128, 128]                    
  6                  -1  1    131840  ultralytics.nn.modules.mamba_yolo.VisionClueM


/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:349: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.op = self.SSM_CLASS(
/content/Mamba-YOLO/ultralytics/nn/modules/mamba_yolo.py:277: FutureWarning: functools.partial will be a method descriptor in future Python versions; wrap it in staticmethod() if you want to preserve the old behavior
  self.ss2d = nn.Sequential(*(self.SSM_CLASS(d_model=self.hidden_dim,


mamba-YOLO-T-k5aug summary: 544 layers, 6215245 parameters, 6215229 gradients, 13.4 GFLOPs

Freezing layer 'model.21.dfl.conv.weight'
AMP: running Automatic Mixed Precision (AMP) checks with YOLOv8n...


...

[OffsetMonitor] epoch 20: blok |tanh(dp)| rata2>0.99: 0/10 | token di luar [-1,1]: 15.2% | token pindah: 96.4%


                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 43/43 [00:05<00:00,  7.33it/s]

                   all        682       1381   0.000191     0.0509   0.000402   0.000126



      Epoch    GPU_mem   box_loss   cls_loss   dfl_loss  Instances       Size


  0%|          | 0/341 [00:00<?, ?it/s]/content/Mamba-YOLO/ultralytics/engine/trainer.py:374: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(self.amp):
/usr/local/lib/python3.13/dist-packages/torch/autograd/graph.py:869: UserWarning: grid_sampler_2d_backward_cuda does not have a deterministic implementation, but you set 'torch.use_deterministic_algorithms(True, warn_only=True)'. You can file an issue at https://github.com/pytorch/pytorch/issues to help us prioritize adding deterministic support for this operation. (Triggered internally at /pytorch/aten/src/ATen/Context.cpp:160.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass
     21/150      8.05G      3.455       6.81       3.04         17        640:  19%|█▉        | 64/341 [00:21<01:36,  2.86it/s]

## 9. Evaluasi + CSV (bergabung dgn hasil resep auto lainnya)

In [ ]:
import pandas as pd
OUT_CSV = f"{DRIVE_ROOT}/analysis/per_class_results_recipe.csv"
best = Path(SAVE_DIR)/"weights"/"best.pt"
assert best.exists(), "best.pt belum ada"
r = YOLO(str(best)).val(data=DATA_YAML, split="val", verbose=False)
base = dict(model=MODEL_LABEL, seed=SEED, ckpt_path=str(best))
rows = [{**base,"class":"ALL","mAP50_95":r.box.map,"mAP50":r.box.map50,
         "precision":r.box.mp,"recall":r.box.mr}]
for i,c in enumerate(NAMES):
    rows.append({**base,"class":c,"mAP50_95":float(r.box.maps[i]),
                 "mAP50":float("nan"),"precision":float("nan"),"recall":float("nan")})
df = pd.DataFrame(rows)
if os.path.exists(OUT_CSV):
    df = pd.concat([pd.read_csv(OUT_CSV), df]).drop_duplicates(["model","seed","class"], keep="last")
df.to_csv(OUT_CSV, index=False); shutil.copy(OUT_CSV, "/content/per_class_results_recipe.csv")
v = df[(df["class"]=="ALL") & (df.model==MODEL_LABEL)].mAP50_95.iloc[0]
print(df[df["class"]=="ALL"][["model","seed","mAP50_95","mAP50","precision","recall"]].to_string(index=False))
print(f"\n  B2-auto (baseline yang benar) : 0.6796")
print(f"  k5-Aug (RUN INI)              : {v:.4f}   ({v-0.6796:+.4f})")
print(f"  P1 di resep ini               : DIVERGEN")
print(f"\n  Kelas elongated (crack, broken-arc) -- fokus hipotesis:")
for c in ["crack","broken-arc"]:
    print(f"    {c:12s} {df[(df.model==MODEL_LABEL)&(df['class']==c)].mAP50_95.iloc[0]:.4f}")
for _src,_dst in [(Path(SAVE_DIR)/"offset_monitor.csv","/content/k5aug_offset_monitor.csv"),
                  (Path(SAVE_DIR)/"results.csv","/content/k5aug_results.csv")]:
    if _src.exists(): shutil.copy(_src,_dst); print("salinan:",_dst)
print(f"\nTersimpan: {OUT_CSV}")
print("KIRIM ke sesi analisis: per_class_results_recipe.csv, k5aug_offset_monitor.csv, k5aug_results.csv")

## 10. Matikan Runtime (Jalankan Terakhir)

In [ ]:
print("Semua proses selesai. Runtime akan di-disconnect otomatis...")
import time
time.sleep(10)
try:
    from google.colab import runtime
    runtime.unassign()
except Exception as e:
    print(f"Gagal disconnect otomatis ({e})")
    print("Matikan manual lewat: Runtime > Disconnect and delete runtime")